# DPO
[Source](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/04_preference-tuning-with-dpo/dpo-from-scratch.ipynb)

## 0 · Goal & config
- **Mac:** local `.venv` kernel; run from `Training/`.
- **Runpod:** keep [setup.ipynb](../Runpod/setup.ipynb) on the local kernel and run **Start**. Here, choose **Select Kernel → Select Another Kernel → Existing Jupyter Server**, paste the URL, then choose **Python 3**. The first code cell sets the folder; device should print `cuda`.
- **Runpod checkpoint:** upload the instruction checkpoint with setup's `upload_checkpoint(...)` before selecting it below.

In [1]:
from pathlib import Path

# Use the copied project folder on Runpod.
pod_folder = Path("/workspace/LLM-fine-tuning/Training")
if pod_folder.is_dir():
    %cd {pod_folder}

#### Imports

In [64]:
from functools import partial

import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs
from reference.data import load_dpo_data
from reference.generate import generate, generate_and_print_sample
from reference.model_io import text_to_token_ids, token_ids_to_text
from reference.models.gpt2 import GPT2Model


#### Model config

In [3]:
# Options: gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M).
CHOOSE_MODEL = "gpt2-medium (355M)"

# ** unpacks dictionary entries; later values win on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}

#### Run folder

In [4]:
folder = Path("runs/dpo/001_baseline")
checkpoint_path = max(Path("runs/instruction/002_finetuned").glob("checkpoint_*.pth"))


#### Device

In [5]:
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("device:", device)

device: mps


#### Tokenizer

In [6]:
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.n_vocab, tokenizer.eot_token

(50257, 50256)

## 1 · Load model
Load the instruction-finetuned checkpoint.

In [7]:
%%capture

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model_config = checkpoint.get("model_config", model_config)

model = GPT2Model(model_config)
model.load_state_dict(checkpoint["model"])
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)
model.eval()
# del checkpoint


#### Policy & Reference model

In [8]:

policy_model = GPT2Model(model_config)
policy_model.load_state_dict(checkpoint["model"])
policy_model.train()
policy_model.to(device)

# only passing parameters of the policy model in optimizer; (don't want to modify the reference model)
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)


reference_model = GPT2Model(model_config)
reference_model.load_state_dict(checkpoint["model"])
reference_model.eval()
reference_model.to(device)

GPT2Model(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MaskedMultiHeadAttn(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MaskedMultiHeadAttn(
        (W_query): Lin

## 2 · Data + DataLoader
Load data, prepare token IDs, and build batches.

In [9]:
data = load_dpo_data()
data[:1]

[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".',
  'rejected': 'The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".',
  'chosen': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]

#### Creating training, validation, and test splits    
Next, we divide the dataset into 3 subsets, 85% training data, 5% validation data, and 10% test data:

In [10]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [11]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}")

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

desired_response = f"### Response:\n{data[0]['chosen']}" # 'rejected'
desired_response

'### Response:\nThe spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'

**Dataset groups by example → batch groups by key.**

### Dataset: list of dictionaries

```text
[
  {"prompt": P1, "chosen": C1, "rejected": R1},
  {"prompt": P2, "chosen": C2, "rejected": R2}
]
```

### After collate: one batch dictionary

```text
{
  "prompt":        [tensor(P1), tensor(P2)],
  "chosen":        tensor([C1, C2]),
  "rejected":      tensor([R1, R2]),
  "chosen_mask":   tensor([mask_C1, mask_C2]),
  "rejected_mask": tensor([mask_R1, mask_R2])
}
```

`P` = prompt tokens; `C/R` = prompt + chosen/rejected response tokens.

**Each tensor row = one example.** Chosen/rejected and their masks have shape `[batch_size, sequence_length]`; `prompt` stays a list of unpadded tensors.

In [12]:
class PreferenceDataset(Dataset):

    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_data = []
        for entry in data:
            prompt = format_input(entry)
            combined_chosen = f"{prompt}\n\n### Response:\n{entry['chosen']}"
            combined_rejected = f"{prompt}\n\n### Response:\n{entry['rejected']}"

            encoded_prompt    = tokenizer.encode(prompt)
            encoded_chosen    = tokenizer.encode(combined_chosen)
            encoded_rejected  = tokenizer.encode(combined_rejected)
            
            self.encoded_data.append({
                "prompt":encoded_prompt,
                "chosen": encoded_chosen,
                "rejected": encoded_rejected
            })
    
    def __getitem__(self, index):
        return self.encoded_data[index]
    
    def __len__(self):
        return len(self.data)



dataset = PreferenceDataset(data, tokenizer)
for key, values in dataset.__getitem__(0).items():
    print(key, len(values))

prompt 48
chosen 74
rejected 80


**Collate:** pad and stack chosen/rejected sequences; leave token IDs unshifted.

**Masks:** `True` = score this token; `False` = ignore it. Padding is ignored. With prompt masking enabled, the prompt and two following newline tokens are also ignored; the response header still counts.

In [13]:
def custom_collate_fn(batch, pad_token = 50256, 
                      allowed_max_length= None, 
                      mask_prompt_tokens = True,  device="cpu"):

    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []
    }

    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max([len(item[key])+1 for item in batch])
            max_length_common = max(max_length_common, current_max)
        

    # Process each item in the batch: padding, tensor, 
    for item in batch:
        prompt_tensor = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt_tensor)
        
        for key in ["chosen", "rejected"]:
            
            # Adjust padding according to the common maximum length
            padded_item = torch.tensor(item[key] + [pad_token] * (max_length_common - len(item[key])))
            batch_data[key].append(padded_item)

            mask = torch.ones(len(padded_item)).bool()
             # Set mask for all padding tokens to False
            mask[len(item[key]):] = False

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:len(prompt_tensor)+2] = False

            batch_data[f"{key}_mask"].append(mask)


    for key in ["chosen", "chosen_mask", "rejected", "rejected_mask"]:
        # Stack all sequences into a tensor for the given key
        torch_stack = torch.stack(batch_data[key])

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
                torch_stack = torch_stack[:,:allowed_max_length]

        batch_data[key] = torch_stack.to(device)

    return batch_data


In [14]:
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)


In [15]:
example_data = data[:2]

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

for batch in example_dataloader:
    break

print("batch.keys:", batch.keys())
batch["prompt"]

batch.keys: dict_keys(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])


[tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545]),
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 18378,   262,  1708,  6827,   329, 23491,
            13,   198,   198, 21017, 23412,    25,   198,  1544,   467,   284,
           262,  3952,   790,  1110,    13])]

In [16]:
def decode_tokens_from_batch(token_ids, tokenizer):
    flat_tensor = token_ids.flatten().tolist()
    print(tokenizer.decode(flat_tensor))

decode_tokens_from_batch(batch["chosen"][0], tokenizer)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [17]:
print("chosen inputs:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)

chosen inputs: torch.Size([81])
chosen mask:   torch.Size([81])


In [18]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".
None


#### Creating training, validation, and test set data loaders

All chosen/rejected sequences within a batch share the same padded length. That length can differ between batches.

In [28]:
num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)


val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

    

## 3 · Loss / objective

**Next-token alignment:** inputs `[A, B, C]` predict targets `[B, C, D]`.

- **Instruction:** the collator creates the shifted inputs/targets.
- **DPO:** `compute_logprobs` aligns `logits[:, :-1, :]` with `labels[:, 1:]`.

`B` = batch size; `T` = token positions; `V` = vocabulary size.

**Policy** = model being trained; **reference** = fixed copy of the starting model. Both score responses already in the dataset.

1. **`compute_logprobs` → one score per response.** In this notebook, select actual next-token log-probabilities and average only positions kept by the mask **within each example**:

   ```text
   Vocabulary scores:       [B, T, V]
   After next-token shift:  [B, T−1, V]
   Actual-token log-probs:  [B, T−1]
   Mean per response:       [B]

   Example 1: kept log-probs [-2, -4] → -3
   Example 2: kept log-probs [-1, -3] → -2
   Returned scores: [-3, -2]              shape [2]
   ```

2. **`compute_dpo_loss` → one batch loss.** Compare each chosen/rejected pair against the reference.

   Here, `logits` = policy's chosen−rejected gap minus reference's gap → `[B]`.

   `-F.logsigmoid(beta * logits)` gives `[B]` pair losses; `losses.mean()` averages **across the batch** → one scalar loss.

3. **`compute_dpo_loss_batch` → connects everything.** Example: **2 examples × 5 token positions**, vocabulary **50,257**.

   ```text
   Training loop
     ↓ batch: chosen/rejected tokens + masks, each [2, 5]
   ┌─ compute_dpo_loss_batch ───────────────────────────┐
   │ Four scoring passes:                               │
   │   Policy:    chosen, rejected                      │
   │   Reference: chosen, rejected (no_grad)            │
   │                                                    │
   │ Each pass:                                         │
   │   Token IDs [2, 5]                                 │
   │       ↓ model(...)                                 │
   │   Logits [2, 5, 50257]                             │
   │       + same token IDs [2, 5] as labels            │
   │       + matching mask  [2, 5]                      │
   │       ↓ compute_logprobs(...)                      │
   │   Scores [2] = one score per example               │
   │                                                    │
   │ After all four passes:                             │
   │   Four score tensors [2] + beta                    │
   │       ↓ compute_dpo_loss(...)                      │
   │   Two pair losses → losses.mean() → one batch loss │
   │   Also returns two average monitoring rewards      │
   └────────────────────────────────────────────────────┘
     ↓ returns (loss, chosen_reward, rejected_reward): 3 numbers
   Training loop: loss updates policy; rewards track progress.
   ```

**Compared with instruction:** `calc_loss_batch` uses `cross_entropy` to average **negative log-probabilities over all nonignored tokens in the batch** (`reduction="mean"` by default). DPO keeps each response score separate until its pair's loss is calculated, then averages the pair losses.

In [45]:
for batch in train_loader:
    for key in batch.keys():
        # Check if it's a tensor or a list
        if hasattr(batch[key], 'shape'):
            print(f"{key}'s shape = {batch[key].shape}")
        else:
            print(f"{key}'s length = {len(batch[key])}")
            print(f"Tensors' shapes in {key} = {[elem.shape[0] for elem in batch[key]]}\n")
    break


prompt's length = 8
Tensors' shapes in prompt = [37, 33, 48, 44, 35, 35, 35, 35]

chosen's shape = torch.Size([8, 71])
rejected's shape = torch.Size([8, 71])
rejected_mask's shape = torch.Size([8, 71])
chosen_mask's shape = torch.Size([8, 71])


In [ ]:
sample_logits = policy_model(batch['chosen'][0].unsqueeze(0).to(device))
# logprobs = torch.log_softmax(sample_logits, dim = -1)

# shape: (batch_size, seq_length, vocab_size)
sample_logits.shape 

torch.Size([1, 71, 50257])

#### compute logprobs

In [ ]:
def compute_logprobs(logits, labels, selection_mask=None):

    # shape: (batch_size, seq_length, vocab_size)
    logits   = logits[:,:-1,:] 
    logprobs = torch.log_softmax(logits, dim = -1)

    # adjust labels - shift by 1
    labels = labels[:,1:].clone()

    selected_log_probs = torch.gather(
        input= logprobs,
        dim = -1,
        index = labels.unsqueeze(-1)).squeeze(-1)

    if selection_mask is not None:
        # leave first token out - same as labels
        selection_mask = selection_mask[:,1:]
        selected_log_probs = selected_log_probs * selection_mask  
        avg_logprobs = selected_log_probs.sum(dim=-1) / selection_mask.sum(dim=-1)
    
    else:
        avg_logprobs = selected_log_probs.mean(dim=-1)

    return avg_logprobs # shape: (batch_size)

### compute_dpo_loss 

- input four [batch_size] tensors: policy chosen/rejected and reference chosen/rejected + beta

- returns:
    - Mean DPO loss.
    - Mean chosen reward.
    - Mean rejected reward.

In [ ]:

def compute_dpo_loss(policy_chosen_logprobs, policy_rejected_logprobs, reference_chosen_logprobs, reference_rejected_logprobs, beta = 0.1):
    
    # shape: batch_size
    policy_logratio    = policy_chosen_logprobs - policy_rejected_logprobs 
    reference_logratio = reference_chosen_logprobs - reference_rejected_logprobs

    logits = (policy_logratio - reference_logratio) * beta
    losses = -F.logsigmoid(logits)

    # use .detach() because these are monitoring values. Keep gradients on the loss
    chosen_rewards   = (policy_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (policy_rejected_logprobs - reference_rejected_logprobs).detach()

    # returns a scalar with shape [] for all 3
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()
    

#### compute_dpo_loss_batch

    - uses compute_logprobs & compute_dpo_loss fns

 - batch keys: (['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])

In [ ]:
(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])

In [ ]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):

    policy_chosen_logprobs      = compute_logprobs(logits = policy_model(batch['chosen']), labels = batch['chosen'], selection_mask=batch['chosen_mask'])
    policy_rejected_logprobs    = compute_logprobs(logits = policy_model(batch['rejected']), labels = batch['rejected'], selection_mask=batch['rejected_mask'])

    with torch.no_grad():
        reference_chosen_logprobs   = compute_logprobs(logits = reference_model(batch['chosen']), labels = batch['chosen'], selection_mask=batch['chosen_mask'])
        reference_rejected_logprobs = compute_logprobs(logits = reference_model(batch['rejected']), labels = batch['rejected'], selection_mask=batch['rejected_mask'])
        
    loss, chosen_reward, rejected_reward = compute_dpo_loss(policy_chosen_logprobs, policy_rejected_logprobs, reference_chosen_logprobs, reference_rejected_logprobs, beta = beta)
    
    # returns a scalar with shape [] for all 3
    return loss, chosen_reward, rejected_reward

#### Next: average across batches - compute_dpo_loss_loader

In [ ]:
# Next: average across batches
 
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):

    if len(data_loader) == 0:
        return (float("nan"),) * 3
    elif num_batches is not None:
        num_batches = min(num_batches, len(data_loader))
    else:
        num_batches = len(data_loader)

    total_loss, total_chosen_reward, total_rejected_reward = 0., 0., 0. 


    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_reward, rejected_reward = compute_dpo_loss_batch(batch, policy_model, reference_model, beta)
            
            total_loss            += loss.item()
            total_chosen_reward   += chosen_reward.item()
            total_rejected_reward += rejected_reward.item() 

        else:
            break

    total_loss            /= num_batches
    total_chosen_reward   /= num_batches
    total_rejected_reward /= num_batches

    return total_loss, total_chosen_reward, total_rejected_reward

#### evaluation wrapper

In [ ]:
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):
    policy_model.eval()

    with torch.no_grad():
        train_loss, train_chosen_reward, train_rejected_reward = compute_dpo_loss_loader(train_loader, policy_model, reference_model, beta, num_batches=eval_iter)
        val_loss, val_chosen_reward, val_rejected_reward       = compute_dpo_loss_loader(val_loader, policy_model, reference_model, beta, num_batches=eval_iter)

    res = {
        "train_loss":train_loss,
        "train_chosen_reward":train_chosen_reward,
        "train_rejected_reward":train_rejected_reward,
        "val_loss":val_loss,
        "val_chosen_reward":val_chosen_reward,
        "val_rejected_reward":val_rejected_reward,
    }
    
    policy_model.train()
    return res
        


## 4 · Generate / predict
Generate a starting response before this training run.

In [63]:
torch.manual_seed(123)
policy_model.eval()
sample_input = format_input(val_data[0])
token_ids = generate(policy_model, 
idx = text_to_token_ids(sample_input, tokenizer).to(device), 
max_new_tokens = 35,
context_length = model_config["context_length"],
eos_id = tokenizer.eot_token)

generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(sample_input):].strip()
print(f"{response_text}")

### Response:
The meal is cooked everyday by the chef.


## 5. Training loop

In [ ]:
def train_model_dpo_simple(
    policy_model, reference_model, optimizer, tokenizer, device,
    train_loader, val_loader, eval_freq, eval_iter, start_context, beta, num_epochs):

    tracking = {
        "train_losses":[],
        "train_chosen_rewards":[],
        "train_rejected_rewards":[],
        "val_losses":[],
        "val_chosen_rewards":[],
        "val_rejected_rewards":[],
        "tokens_seen":[]
    }

    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        policy_model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            loss, chosen_reward, rejected_reward = compute_dpo_loss_batch(batch, policy_model, reference_model, beta)
            loss.backward()
            optimizer.step()

            global_step += 1
            tokens_seen += batch['chosen'].numel()
            
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter)
                tracking['train_losses'].append(res['train_loss'])
                tracking['train_chosen_rewards'].append(res['train_chosen_reward'])
                tracking['train_rejected_rewards'].append(res['train_rejected_reward'])
                tracking['val_losses'].append(res['val_loss'])
                tracking['val_chosen_rewards'].append(res['val_chosen_reward'])
                tracking['val_rejected_rewards'].append(res['val_rejected_reward'])
                tracking['tokens_seen'].append(tokens_seen)
                train_reward_margin = res["train_chosen_reward"] - res["train_rejected_reward"]
                val_reward_margin   = res["val_chosen_reward"] - res["val_rejected_reward"]

                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                    f"Train reward margins {train_reward_margin:.3f}, "
                    f"Val reward margins {val_reward_margin:.3f}"
                )

        # Print a sample text after each epoch
        generate_and_print_sample(
            model=policy_model,
            tokenizer=tokenizer,
            device=device,
            start_context=start_context
        )

    return tracking
    